# Uber Data Pipeline (Fixed Version)

Refined version of the original notebook:
- keeps the original star-schema ETL idea
- removes noisy outputs and repeated steps
- adds weather data ingestion from Open-Meteo Archive API
- creates a weather-aware trip mart for analysis


In [1]:
import io
import pandas as pd
import requests


In [2]:
from pathlib import Path
import pyarrow as pa

TAXI_YEAR = 2025
TAXI_MONTH = 1
TAXI_URL = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{TAXI_YEAR}-{TAXI_MONTH:02d}.parquet"

try:
    pa.unregister_extension_type("pandas.period")
except (KeyError, AttributeError):
    pass

df = pd.read_parquet(TAXI_URL)


df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

df = df.drop_duplicates().dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"]).reset_index(drop=True)
df["trip_id"] = df.index

df["trip_duration_min"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60

df = df[(df["trip_duration_min"] > 0) & (df["fare_amount"] >= 0) & (df["trip_distance"] > 0)].reset_index(drop=True)
df["trip_id"] = df.index

df.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_id,trip_duration_min
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0,0,8.350000
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,1,2.550000
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0,2,1.950000
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0,3,5.566667
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0,4,3.533333


In [3]:
df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee', 'trip_id', 'trip_duration_min'],
      dtype='object')

In [4]:
datetime_dim = df[["tpep_pickup_datetime", "tpep_dropoff_datetime"]].copy()
datetime_dim["pick_hour"] = datetime_dim["tpep_pickup_datetime"].dt.hour
datetime_dim["pick_day"] = datetime_dim["tpep_pickup_datetime"].dt.day
datetime_dim["pick_month"] = datetime_dim["tpep_pickup_datetime"].dt.month
datetime_dim["pick_year"] = datetime_dim["tpep_pickup_datetime"].dt.year
datetime_dim["pick_weekday"] = datetime_dim["tpep_pickup_datetime"].dt.weekday

datetime_dim["drop_hour"] = datetime_dim["tpep_dropoff_datetime"].dt.hour
datetime_dim["drop_day"] = datetime_dim["tpep_dropoff_datetime"].dt.day
datetime_dim["drop_month"] = datetime_dim["tpep_dropoff_datetime"].dt.month
datetime_dim["drop_year"] = datetime_dim["tpep_dropoff_datetime"].dt.year
datetime_dim["drop_weekday"] = datetime_dim["tpep_dropoff_datetime"].dt.weekday

datetime_dim["datetime_id"] = datetime_dim.index
datetime_dim = datetime_dim[[
    "datetime_id", "tpep_pickup_datetime", "pick_hour", "pick_day", "pick_month", "pick_year", "pick_weekday",
    "tpep_dropoff_datetime", "drop_hour", "drop_day", "drop_month", "drop_year", "drop_weekday"
]]
datetime_dim.head()


,datetime_id,tpep_pickup_datetime,pick_hour,pick_day,pick_month,pick_year,pick_weekday,tpep_dropoff_datetime,drop_hour,drop_day,drop_month,drop_year,drop_weekday
0,0,2025-01-01 00:18:38,0,1,1,2025,2,2025-01-01 00:26:59,0,1,1,2025,2
1,1,2025-01-01 00:32:40,0,1,1,2025,2,2025-01-01 00:35:13,0,1,1,2025,2
2,2,2025-01-01 00:44:04,0,1,1,2025,2,2025-01-01 00:46:01,0,1,1,2025,2
3,3,2025-01-01 00:14:27,0,1,1,2025,2,2025-01-01 00:20:01,0,1,1,2025,2
4,4,2025-01-01 00:21:34,0,1,1,2025,2,2025-01-01 00:25:06,0,1,1,2025,2


In [5]:
passenger_count_dim = df[["passenger_count"]].copy()
passenger_count_dim["passenger_count_id"] = passenger_count_dim.index
passenger_count_dim = passenger_count_dim[["passenger_count_id", "passenger_count"]]

trip_distance_dim = df[["trip_distance"]].copy()
trip_distance_dim["trip_distance_id"] = trip_distance_dim.index
trip_distance_dim = trip_distance_dim[["trip_distance_id", "trip_distance"]]


In [6]:
rate_code_type = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau or Westchester",
    5: "Negotiated fare",
    6: "Group ride"
}

rate_code_dim = df[["RatecodeID"]].copy()
rate_code_dim["rate_code_id"] = rate_code_dim.index
rate_code_dim["rate_code_name"] = rate_code_dim["RatecodeID"].map(rate_code_type).fillna("Unknown")
rate_code_dim = rate_code_dim[["rate_code_id", "RatecodeID", "rate_code_name"]]
rate_code_dim.head()


,rate_code_id,RatecodeID,rate_code_name
0,0,1.0,Standard rate
1,1,1.0,Standard rate
2,2,1.0,Standard rate
3,3,1.0,Standard rate
4,4,1.0,Standard rate


In [7]:
pickup_location_dim = df[["PULocationID"]].drop_duplicates().reset_index(drop=True).copy()
pickup_location_dim["pickup_location_id"] = pickup_location_dim.index
pickup_location_dim = pickup_location_dim[["pickup_location_id", "PULocationID"]]

dropoff_location_dim = df[["DOLocationID"]].drop_duplicates().reset_index(drop=True).copy()
dropoff_location_dim["dropoff_location_id"] = dropoff_location_dim.index
dropoff_location_dim = dropoff_location_dim[["dropoff_location_id", "DOLocationID"]]


In [8]:
payment_type_name = {
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip"
}

payment_type_dim = df[["payment_type"]].copy()
payment_type_dim["payment_type_id"] = payment_type_dim.index
payment_type_dim["payment_type_name"] = payment_type_dim["payment_type"].map(payment_type_name).fillna("Unknown")
payment_type_dim = payment_type_dim[["payment_type_id", "payment_type", "payment_type_name"]]


In [9]:
fact_table = (
    df.merge(passenger_count_dim, left_on="trip_id", right_on="passenger_count_id")
      .merge(trip_distance_dim, left_on="trip_id", right_on="trip_distance_id")
      .merge(rate_code_dim, left_on="trip_id", right_on="rate_code_id")
      .merge(pickup_location_dim, on="PULocationID")
      .merge(dropoff_location_dim, on="DOLocationID")
      .merge(datetime_dim, left_on="trip_id", right_on="datetime_id")
      .merge(payment_type_dim, left_on="trip_id", right_on="payment_type_id")
)

fact_table = fact_table[[
    "trip_id", "VendorID", "datetime_id", "passenger_count_id", "trip_distance_id", "rate_code_id",
    "store_and_fwd_flag", "pickup_location_id", "dropoff_location_id", "payment_type_id",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount"
]]
fact_table.head()


,trip_id,VendorID,datetime_id,passenger_count_id,trip_distance_id,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type_id,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,0,1,0,0,0,0,N,0,0,0,10.0,3.5,0.5,3.00,0.0,1.0,18.00
1,1,1,1,1,1,1,N,1,0,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12
2,2,1,2,2,2,2,N,2,1,2,5.1,3.5,0.5,2.00,0.0,1.0,12.10
3,3,2,3,3,3,3,N,3,2,3,7.2,1.0,0.5,0.00,0.0,1.0,9.70
4,4,2,4,4,4,4,N,3,3,4,5.8,1.0,0.5,0.00,0.0,1.0,8.30


## Weather ingestion and processing (Open-Meteo)

The original pipeline did not include weather. The cells below map `PULocationID` to borough using the official TLC zone lookup and then join hourly weather on `pickup_hour + borough`.


In [10]:
ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

zone_lookup_df = pd.read_csv(ZONE_LOOKUP_URL)
zone_lookup_df = zone_lookup_df[["LocationID", "Borough", "Zone", "service_zone"]]

borough_anchors = {
    "Manhattan": (40.7831, -73.9712),
    "Brooklyn": (40.6782, -73.9442),
    "Queens": (40.7282, -73.7949),
    "Bronx": (40.8448, -73.8648),
    "Staten Island": (40.5795, -74.1502),
}

start_date = df["tpep_pickup_datetime"].min().date().isoformat()
end_date = df["tpep_pickup_datetime"].max().date().isoformat()

weather_frames = []
for borough, (latitude, longitude) in borough_anchors.items():
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,precipitation,rain,snowfall,windspeed_10m,weathercode",
        "timezone": "America/New_York",
    }

    weather_resp = requests.get(OPEN_METEO_URL, params=params, timeout=60)
    weather_resp.raise_for_status()
    weather_payload = weather_resp.json()
    borough_weather_df = pd.DataFrame(weather_payload["hourly"])
    borough_weather_df["borough"] = borough
    weather_frames.append(borough_weather_df)

weather_df = pd.concat(weather_frames, ignore_index=True)
weather_df["weather_ts_hour"] = pd.to_datetime(weather_df["time"], errors="coerce")
weather_df = weather_df.rename(columns={
    "precipitation": "precipitation_mm",
    "rain": "rain_mm",
    "snowfall": "snowfall_cm",
    "windspeed_10m": "wind_speed",
})

weather_df["is_rain"] = weather_df["rain_mm"].fillna(0) > 0
weather_df["is_snow"] = weather_df["snowfall_cm"].fillna(0) > 0
weather_df["weather_severity"] = "clear"
weather_df.loc[(weather_df["precipitation_mm"] > 0) | (weather_df["snowfall_cm"] > 0), "weather_severity"] = "light"
weather_df.loc[(weather_df["precipitation_mm"] >= 2) | (weather_df["snowfall_cm"] >= 1), "weather_severity"] = "moderate"
weather_df.loc[(weather_df["precipitation_mm"] >= 8) | (weather_df["snowfall_cm"] >= 3), "weather_severity"] = "severe"

weather_df = weather_df[[
    "borough", "weather_ts_hour", "temperature_2m", "precipitation_mm", "rain_mm", "snowfall_cm", "wind_speed", "weathercode",
    "is_rain", "is_snow", "weather_severity"
]]
weather_df.head()


,borough,weather_ts_hour,temperature_2m,precipitation_mm,rain_mm,snowfall_cm,wind_speed,weathercode,is_rain,is_snow,weather_severity
0,Manhattan,2024-12-31 00:00:00,7.2,0.0,0.0,0.0,12.3,0,False,False,clear
1,Manhattan,2024-12-31 01:00:00,6.6,0.0,0.0,0.0,11.6,0,False,False,clear
2,Manhattan,2024-12-31 02:00:00,5.7,0.0,0.0,0.0,9.5,0,False,False,clear
3,Manhattan,2024-12-31 03:00:00,4.9,0.0,0.0,0.0,8.5,0,False,False,clear
4,Manhattan,2024-12-31 04:00:00,4.2,0.0,0.0,0.0,7.7,0,False,False,clear


In [11]:
trip_weather_df = df.copy()
trip_weather_df = trip_weather_df.merge(zone_lookup_df, left_on="PULocationID", right_on="LocationID", how="left")
trip_weather_df = trip_weather_df.rename(columns={"Borough": "borough", "Zone": "pickup_zone"})
trip_weather_df["pickup_hour"] = trip_weather_df["tpep_pickup_datetime"].dt.floor("h")

trip_weather_df = trip_weather_df.merge(
    weather_df,
    left_on=["pickup_hour", "borough"],
    right_on=["weather_ts_hour", "borough"],
    how="left"
)
trip_weather_df["fare_per_mile"] = trip_weather_df["fare_amount"] / trip_weather_df["trip_distance"]

trip_weather_df[[
    "trip_id", "tpep_pickup_datetime", "borough", "pickup_zone", "trip_distance", "fare_amount", "total_amount", "fare_per_mile", "trip_duration_min",
    "temperature_2m", "precipitation_mm", "is_rain", "is_snow", "weather_severity"
]].head()


,trip_id,tpep_pickup_datetime,borough,pickup_zone,trip_distance,fare_amount,trip_duration_min,temperature_2m,precipitation_mm,is_rain,is_snow,weather_severity
0,0,2025-01-01 00:18:38,Manhattan,Sutton Place/Turtle Bay North,1.60,10.0,8.350000,7.6,3.7,True,False,moderate
1,1,2025-01-01 00:32:40,Manhattan,Upper East Side North,0.50,5.1,2.550000,7.6,3.7,True,False,moderate
2,2,2025-01-01 00:44:04,Manhattan,Lenox Hill West,0.60,5.1,1.950000,7.6,3.7,True,False,moderate
3,3,2025-01-01 00:14:27,Manhattan,Washington Heights South,0.52,7.2,5.566667,7.6,3.7,True,False,moderate
4,4,2025-01-01 00:21:34,Manhattan,Washington Heights South,0.66,5.8,3.533333,7.6,3.7,True,False,moderate


In [12]:
weather_hourly_mart = (
    trip_weather_df.groupby(["pickup_hour", "borough", "weather_severity"], dropna=False)
    .agg(
        trip_count=("trip_id", "count"),
        avg_fare=("fare_amount", "mean"),
        avg_total_amount=("total_amount", "mean"),
        avg_duration_min=("trip_duration_min", "mean"),
        avg_distance=("trip_distance", "mean"),
        avg_fare_per_mile=("fare_per_mile", "mean"),
        avg_precipitation=("precipitation_mm", "mean"),
        avg_temp=("temperature_2m", "mean")
    )
    .reset_index()
    .sort_values(["pickup_hour", "borough", "weather_severity"])
)

weather_hourly_mart.head()


,pickup_hour,borough,weather_severity,trip_count,avg_fare,avg_duration_min,avg_distance,avg_precipitation,avg_temp
0,2024-12-31 20:00:00,Manhattan,clear,3,20.266667,19.427778,2.720000,0.0,7.7
1,2024-12-31 21:00:00,Manhattan,clear,2,15.950000,12.883333,2.920000,0.0,7.6
2,2024-12-31 21:00:00,Queens,clear,1,7.900000,5.283333,1.120000,0.0,8.3
3,2024-12-31 23:00:00,Manhattan,moderate,11,15.536364,13.798485,2.601818,2.5,7.0
4,2024-12-31 23:00:00,Queens,moderate,4,32.525000,27.950000,8.267500,3.9,8.2


In [ ]:
# Optional exports
# fact_table.to_csv("fact_table.csv", index=False)
# weather_df.to_csv("weather_dim.csv", index=False)
# weather_hourly_mart.to_csv("weather_hourly_mart.csv", index=False)
